# 470 — Concatenated ROIs on exemplar ERSPs

Validation figure: for a hand-picked list of **information-rich** contacts, build the
**concatenated `[audio | picture | reading]`** ERSP (the triptych — *not* one map per condition)
and **overlay the concatenated role ROIs** (`roi_config_concatenated.py`). The unique
(block × time × frequency) box regions are drawn as outlines coloured by frequency band, with
block dividers and per-block response-onset lines — so you see exactly where the role templates
sit on real cross-condition activity.

Each `NAMES` entry identifies a **contact** (pid + electrode); all three conditions are loaded for
it (contacts missing any condition are skipped). `GRID='full'` uses the native 129×900 triptych.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


In [ ]:
NAMES = [
    "EL045_reading_WM_ERSP_pH_L11_TN_CLEAN.png",
    "PAT_3415_reading_WM_ERSP_GE4_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_OPG1_TN_CLEAN.png",
    "PAT_6704_reading_WM_ERSP_TPD5_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_IAG7_TN_CLEAN.png",
    "EL043_reading_WM_ERSP_iSMG3_TN_CLEAN.png",
    "EL042_reading_WM_ERSP_STG_R4_TN_CLEAN.png",
    "PAT_3415_reading_WM_ERSP_OS2_TN_CLEAN.png",
    "EL038_reading_WM_ERSP_aI_R8_TN_CLEAN.png",
    "EL033_reading_WM_ERSP_PHG_R12_TN_CLEAN.png",
    "PAT_3390_reading_WM_ERSP_CPG15_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_IMG9_TN_CLEAN.png",
    "PAT_6854_reading_WM_ERSP_IAG3_TN_CLEAN.png",
    "EL030_reading_WM_ERSP_A_L15_TN_CLEAN.png",
    "PAT_5533_reading_WM_ERSP_OFD5_TN_CLEAN.png",
    "EL045_reading_WM_ERSP_pH_L15_TN_CLEAN.png",
    "EL035_audio_WM_ERSP_TTG_R1_TN_CLEAN.png",
    "EL045_audio_WM_ERSP_aH_L12_TN_CLEAN.png",
]
GRID = 'full'                        # 'full' = native 129x900 triptych
print(len(NAMES), 'names · grid:', GRID)


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
run_dir = P.new_run_dir('roi_on_concat')
ok = miss = 0
seen = set()
for name in NAMES:
    cid = P._contact_id_from_name(name)          # (pid, mid) — condition-independent
    if cid is None or cid in seen:
        continue
    seen.add(cid)
    try:
        concat = P.load_concat_ersp(INPUT_DIR, name)     # (129, 3*300) [audio|picture|reading]
    except (FileNotFoundError, ValueError) as e:
        print('  [skip]', e); miss += 1; continue
    label = f'{cid[0]} {cid[1].split("_ERSP_")[-1]}'
    stem = f'{cid[0]}_{cid[1]}'.replace('/', '_')
    P.plot_roles_on_concat(concat, grid=GRID, label=label, out_png=run_dir / f'{stem}.png')
    plt.close('all')
    display(Image(filename=str(run_dir / f'{stem}.png')))
    ok += 1
print(f'\ndone: {ok} contacts plotted, {miss} skipped (missing a condition) · saved -> {run_dir}')
